# Logistics Data Warehouse — ETL Notebook
**Course:** Data Warehousing and Visualisation  
**Author:** Soelema Remenda Pieka  
**University:** Università della Calabria  
**Academic Year:** 2025/2026  

---

This notebook implements the full ETL pipeline for the logistics data warehouse 
designed in Phase 1. The star schema, all dimension and fact table definitions, 
and all design decisions (attribute tree editing steps, derived measures) are 
documented in the project report.

**Dataset:** Synthetic Logistics Operations Database (2022–2024)  
**Source:** https://www.kaggle.com/datasets/yogape/logistics-operations-database

---
## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import os

# Paths
RAW_PATH = 'data/raw/'
OUTPUT_PATH = 'data/output/'

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Setup complete")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Setup complete
Pandas version: 2.2.2
NumPy version: 1.26.4


---
# EXTRACT
---

## 1. Load Raw CSVs
*Phase 1 reference: Section 1.1 — Data Source Description*  
Load all 10 selected source files into pandas DataFrames.

In [2]:
# Load all 10 source CSV files
drivers = pd.read_csv(RAW_PATH + 'drivers.csv')
trucks = pd.read_csv(RAW_PATH + 'trucks.csv')
trailers = pd.read_csv(RAW_PATH + 'trailers.csv')
customers = pd.read_csv(RAW_PATH + 'customers.csv')
facilities = pd.read_csv(RAW_PATH + 'facilities.csv')
routes = pd.read_csv(RAW_PATH + 'routes.csv')
loads = pd.read_csv(RAW_PATH + 'loads.csv')
trips = pd.read_csv(RAW_PATH + 'trips.csv')
delivery_events = pd.read_csv(RAW_PATH + 'delivery_events.csv')
fuel_purchases = pd.read_csv(RAW_PATH + 'fuel_purchases.csv')

print("All files loaded successfully")
print(f"\nRow counts:")
print(f"  drivers:         {len(drivers):>10,}")
print(f"  trucks:          {len(trucks):>10,}")
print(f"  trailers:        {len(trailers):>10,}")
print(f"  customers:       {len(customers):>10,}")
print(f"  facilities:      {len(facilities):>10,}")
print(f"  routes:          {len(routes):>10,}")
print(f"  loads:           {len(loads):>10,}")
print(f"  trips:           {len(trips):>10,}")
print(f"  delivery_events: {len(delivery_events):>10,}")
print(f"  fuel_purchases:  {len(fuel_purchases):>10,}")

All files loaded successfully

Row counts:
  drivers:                150
  trucks:                 120
  trailers:               180
  customers:              200
  facilities:              50
  routes:                  58
  loads:               85,410
  trips:               85,410
  delivery_events:    170,820
  fuel_purchases:     196,442


In [3]:
# Dictionary of all dataframes for easy iteration
dataframes = {
    'drivers': drivers,
    'trucks': trucks,
    'trailers': trailers,
    'customers': customers,
    'facilities': facilities,
    'routes': routes,
    'loads': loads,
    'trips': trips,
    'delivery_events': delivery_events,
    'fuel_purchases': fuel_purchases
}

# Print shape and column names for each dataframe
for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Dtypes:\n{df.dtypes}")


  DRIVERS
  Shape: (150, 12)
  Columns: ['driver_id', 'first_name', 'last_name', 'hire_date', 'termination_date', 'license_number', 'license_state', 'date_of_birth', 'home_terminal', 'employment_status', 'cdl_class', 'years_experience']
  Dtypes:
driver_id            object
first_name           object
last_name            object
hire_date            object
termination_date     object
license_number       object
license_state        object
date_of_birth        object
home_terminal        object
employment_status    object
cdl_class            object
years_experience      int64
dtype: object

  TRUCKS
  Shape: (120, 11)
  Columns: ['truck_id', 'unit_number', 'make', 'model_year', 'vin', 'acquisition_date', 'acquisition_mileage', 'fuel_type', 'tank_capacity_gallons', 'status', 'home_terminal']
  Dtypes:
truck_id                 object
unit_number               int64
make                     object
model_year                int64
vin                      object
acquisition_date         ob

---
# TRANSFORM
---

## 2. Data Inspection
*Phase 1 reference: Section 1.2 — Reconciled Database Schema*  
Verify that the raw data matches the E/R schema designed in Phase 1.

In [4]:
# Quick inspection summary for all dataframes
# Shows shape, missing values, duplicates and basic stats for each table

for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(f"  Shape:            {df.shape}")
    print(f"  Missing values:   {df.isnull().sum().sum()}")
    print(f"  Duplicate rows:   {df.duplicated().sum()}")
    print(f"  Duplicate IDs:    {df.duplicated(subset=df.columns[0]).sum()}")


  DRIVERS
  Shape:            (150, 12)
  Missing values:   124
  Duplicate rows:   0
  Duplicate IDs:    0

  TRUCKS
  Shape:            (120, 11)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  TRAILERS
  Shape:            (180, 9)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  CUSTOMERS
  Shape:            (200, 8)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  FACILITIES
  Shape:            (50, 9)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  ROUTES
  Shape:            (58, 9)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  LOADS
  Shape:            (85410, 12)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  TRIPS
  Shape:            (85410, 12)
  Missing values:   5066
  Duplicate rows:   0
  Duplicate IDs:    0

  DELIVERY_EVENTS
  Shape:            (170820, 11)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  FUEL_PURCHASES
  Sha

In [5]:
# Detailed missing value inspection for tables with nulls

for name in ['drivers', 'trips', 'fuel_purchases']:
    df = dataframes[name]
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n{'='*50}")
    print(f"  {name.upper()} — Missing Values")
    print(f"{'='*50}")
    print(missing)
    print(f"  Total missing: {missing.sum()} / {len(df) * len(df.columns)} cells")


  DRIVERS — Missing Values
termination_date    124
dtype: int64
  Total missing: 124 / 1800 cells

  TRIPS — Missing Values
driver_id     1714
truck_id      1672
trailer_id    1680
dtype: int64
  Total missing: 5066 / 1024920 cells

  FUEL_PURCHASES — Missing Values
truck_id     3880
driver_id    3988
dtype: int64
  Total missing: 7868 / 2160862 cells


In [6]:
# Check if missing driver, truck, trailer overlap on the same trips
missing_trips = trips[trips['driver_id'].isnull() | 
                      trips['truck_id'].isnull() | 
                      trips['trailer_id'].isnull()]

print(f"Trips missing at least one assignment: {len(missing_trips)}")
print(f"\nMissing combinations:")
print(missing_trips[['driver_id','truck_id','trailer_id']].isnull().sum())
print(f"\nTrips missing all three: {missing_trips[['driver_id','truck_id','trailer_id']].isnull().all(axis=1).sum()}")

Trips missing at least one assignment: 4952

Missing combinations:
driver_id     1714
truck_id      1672
trailer_id    1680
dtype: int64

Trips missing all three: 0


In [7]:
# Check if trips with missing assignments still have financial data via loads
missing_trips = trips[trips['driver_id'].isnull() | 
                      trips['truck_id'].isnull() | 
                      trips['trailer_id'].isnull()]

# Check their load_ids are valid
valid_loads = missing_trips['load_id'].isin(loads['load_id']).sum()
print(f"Missing assignment trips with valid load_id: {valid_loads} / {len(missing_trips)}")

Missing assignment trips with valid load_id: 4952 / 4952


In [8]:
# Check fuel purchases with missing truck_id or driver_id
missing_fuel = fuel_purchases[fuel_purchases['truck_id'].isnull() | 
                              fuel_purchases['driver_id'].isnull()]

print(f"Fuel purchases missing truck_id or driver_id: {len(missing_fuel)}")

# Check if they still have valid trip_id for aggregation
valid_trips = missing_fuel['trip_id'].isin(trips['trip_id']).sum()
print(f"Of those, with valid trip_id: {valid_trips} / {len(missing_fuel)}")

# Check if gallons and total_cost are intact
print(f"\nMissing gallons in affected rows: {missing_fuel['gallons'].isnull().sum()}")
print(f"Missing total_cost in affected rows: {missing_fuel['total_cost'].isnull().sum()}")

Fuel purchases missing truck_id or driver_id: 7774
Of those, with valid trip_id: 7774 / 7774

Missing gallons in affected rows: 0
Missing total_cost in affected rows: 0


In [9]:
# Check 1 — trip_status values
print("Trip status values:")
print(trips['trip_status'].value_counts())

# Check 2 — state to city hypothesis in routes
print("\nOrigin states with more than 1 city:")
print(routes.groupby('origin_state')['origin_city'].nunique().sort_values(ascending=False).head(10))

print("\nDestination states with more than 1 city:")
print(routes.groupby('destination_state')['destination_city'].nunique().sort_values(ascending=False).head(10))

Trip status values:
trip_status
Completed    85410
Name: count, dtype: int64

Origin states with more than 1 city:
origin_state
TX    2
AZ    1
NV    1
TN    1
PA    1
OR    1
OH    1
NY    1
NC    1
CO    1
Name: origin_city, dtype: int64

Destination states with more than 1 city:
destination_state
TX    2
CA    1
CO    1
TN    1
PA    1
OR    1
OH    1
NY    1
NV    1
NC    1
Name: destination_city, dtype: int64


In [10]:
# Check Texas cities in routes
print("Texas origin cities:")
print(routes[routes['origin_state'] == 'TX']['origin_city'].unique())

print("\nTexas destination cities:")
print(routes[routes['destination_state'] == 'TX']['destination_city'].unique())

Texas origin cities:
['Dallas' 'Houston']

Texas destination cities:
['Dallas' 'Houston']


## 3. Data Quality Assessment (ISO 25012)
*Phase 1 reference: Section 2.2 — Data Quality Assessment*

*Framework: DQAReport class adapted from course materials (Oro, 2025)*  
*Standard: ISO/IEC 25012:2008*

The DQA is performed on all 10 source tables before any transformation.
Each table is assessed across 6 quality dimensions:

| Dimension | Question |
|---|---|
| Completeness | Are all required values present? |
| Uniqueness | Are primary keys free from duplicates? |
| Validity | Do values conform to domain rules? |
| Consistency | Are values coherent across fields? |
| Timeliness | Are dates within the expected 2022–2024 window? |
| Accuracy | Are numeric values free from statistical outliers? |

**Scoring:** 🟢 ≥ 0.95 Acceptable · 🟡 0.80–0.95 Warning · 🔴 < 0.80 Critical

In [11]:
# Install required libraries for DQA visualization
# missingno: missing value matrix plots
# pandas and matplotlib already available in base conda environment

import subprocess
subprocess.run(['pip', 'install', 'missingno', '--quiet'])

import missingno as msno
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from datetime import datetime

print("✅ Dependencies loaded")

✅ Dependencies loaded


### 3.1 DQAReport Class

The `DQAReport` class implements the ISO 25012 framework as taught in the course
(Oro, 2025). It is extended here with a sixth dimension — **Accuracy** — using
z-score outlier detection on numeric columns, as introduced in the Advanced
section of the course materials.

Design principles:
- **Repeatable**: same rules produce same scores on every run
- **Scored (0–1)**: enables comparison across tables and dimensions
- **Row-level flags**: boolean Series per dimension feeds directly into the cleaning pipeline
- **Chainable API**: methods return `self` for fluent interface

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# DQAReport — ISO 25012 Data Quality Assessment
# Adapted from course Colab 1 (Oro, 2025)
# Extended with Accuracy dimension (z-score outlier detection)
# ─────────────────────────────────────────────────────────────────────────────

class DQAReport:
    """
    Structured, repeatable Data Quality Assessment Report.
    Evaluates ISO 25012 quality dimensions and produces a scored scorecard.
    
    Dimensions covered:
        1. Completeness  — are all required values present?
        2. Uniqueness    — are primary keys free from duplicates?
        3. Validity      — do values conform to domain rules?
        4. Consistency   — are values coherent across fields?
        5. Timeliness    — are dates within the expected window?
        6. Accuracy      — are numeric values free from outliers? [extension]
    """

    def __init__(self, df: pd.DataFrame, table_name: str, primary_key: str = None):
        # Store a copy of the dataframe to avoid modifying the original
        self.df         = df.copy()
        self.table_name = table_name
        self.pk         = primary_key
        # results: dimension -> {score, issues, details}
        self.results    = {}
        # flags: dimension -> boolean Series (True = issue row)
        self.flags      = {}

    # ── DIMENSION 1: Completeness ─────────────────────────────────────────
    def check_completeness(self, required_cols: list = None):
        """
        Measures the proportion of non-missing values across required columns.
        Score = 1 - (total missing / total values evaluated)
        """
        cols           = required_cols or self.df.columns.tolist()
        missing_counts = self.df[cols].isnull().sum()
        total_values   = len(self.df) * len(cols)
        total_missing  = missing_counts.sum()
        score          = 1 - (total_missing / total_values)

        # Flag any row that has at least one missing value in the required columns
        flag_rows = self.df[cols].isnull().any(axis=1)
        self.flags['completeness'] = flag_rows

        self.results['completeness'] = {
            'score'  : round(score, 4),
            'issues' : int(total_missing),
            'details': f"Missing per column: {missing_counts[missing_counts > 0].to_dict()}"
        }
        return self

    # ── DIMENSION 2: Uniqueness ───────────────────────────────────────────
    def check_uniqueness(self, key_cols: list = None):
        """
        Measures the proportion of non-duplicate rows on the primary key.
        Score = 1 - (duplicate rows / total rows)
        """
        cols  = key_cols or ([self.pk] if self.pk else self.df.columns.tolist())
        dup   = self.df.duplicated(subset=cols, keep=False)
        score = 1 - (dup.sum() / len(self.df))

        self.flags['uniqueness'] = dup
        self.results['uniqueness'] = {
            'score'  : round(score, 4),
            'issues' : int(dup.sum()),
            'details': f"Duplicate rows on {cols}: {int(dup.sum())}"
        }
        return self

    # ── DIMENSION 3: Validity ─────────────────────────────────────────────
    def check_validity(self, rules: dict):
        """
        Measures the proportion of rows passing all domain rules.
        rules: {col_name: lambda Series -> bool Series}  True = VALID
        Score = 1 - (rows with at least one violation / total rows)
        """
        all_invalid  = pd.Series(False, index=self.df.index)
        rule_details = []

        for col, rule_fn in rules.items():
            if col not in self.df.columns:
                continue
            # Apply rule only to non-null values — nulls are handled by completeness
            valid_mask   = rule_fn(self.df[col])
            invalid_mask = ~valid_mask & self.df[col].notna()
            all_invalid |= invalid_mask
            rule_details.append(f"{col}: {int(invalid_mask.sum())} invalid")

        score = 1 - (all_invalid.sum() / len(self.df))
        self.flags['validity'] = all_invalid
        self.results['validity'] = {
            'score'  : round(score, 4),
            'issues' : int(all_invalid.sum()),
            'details': " | ".join(rule_details) if rule_details else "No violations"
        }
        return self

    # ── DIMENSION 4: Consistency ──────────────────────────────────────────
    def check_consistency(self, rules: list):
        """
        Measures the proportion of rows satisfying all cross-column rules.
        rules: list of callables  lambda df -> bool Series  True = CONSISTENT
        Score = 1 - (inconsistent rows / total rows)
        """
        all_inconsistent = pd.Series(False, index=self.df.index)

        for rule_fn in rules:
            # Each rule returns True for consistent rows
            inconsistent      = ~rule_fn(self.df)
            all_inconsistent |= inconsistent

        score = 1 - (all_inconsistent.sum() / len(self.df))
        self.flags['consistency'] = all_inconsistent
        self.results['consistency'] = {
            'score'  : round(score, 4),
            'issues' : int(all_inconsistent.sum()),
            'details': f"{int(all_inconsistent.sum())} rows violate at least one consistency rule"
        }
        return self

    # ── DIMENSION 5: Timeliness ───────────────────────────────────────────
    def check_timeliness(self, date_col: str, max_age_days: int = 365,
                         allow_future: bool = False):
        """
        Measures the proportion of dates within the acceptable time window.
        Stale = older than max_age_days from today
        Future = date is after today (unless allow_future=True)
        Score = 1 - (stale + future rows / total rows)
        
        Note: for this project, max_age_days=1500 and allow_future=True
        are used to correctly assess the 2022-2024 simulation window
        without false stale-date flags from the current date (2026).
        """
        if date_col not in self.df.columns:
            return self

        now    = pd.Timestamp.now()
        col    = pd.to_datetime(self.df[date_col], errors='coerce')
        stale  = (now - col).dt.days > max_age_days
        future = col > now if not allow_future else pd.Series(False, index=col.index)
        flag   = stale | future
        score  = 1 - (flag.sum() / len(self.df))

        self.flags['timeliness'] = flag
        self.results['timeliness'] = {
            'score'  : round(score, 4),
            'issues' : int(flag.sum()),
            'details': f"Future dates: {int(future.sum())} | Stale (>{max_age_days}d): {int(stale.sum())}"
        }
        return self

    # ── DIMENSION 6: Accuracy (extension) ────────────────────────────────
    def check_accuracy(self, numeric_cols: list, z_threshold: float = 3.0):
        """
        Extension of the course framework — introduced as Advanced in Oro (2025).
        Detects statistical outliers using z-score on numeric columns.
        A row is flagged if any column has |z-score| > z_threshold.
        Score = 1 - (outlier rows / total rows)
        """
        all_outliers = pd.Series(False, index=self.df.index)
        col_details  = []

        for col in numeric_cols:
            if col not in self.df.columns:
                continue
            # Convert to numeric, coerce non-numeric to NaN
            series = pd.to_numeric(self.df[col], errors='coerce')
            mean   = series.mean()
            std    = series.std()

            # Skip columns with no variance
            if std == 0 or pd.isna(std):
                continue

            z_scores = ((series - mean) / std).abs()
            outliers = (z_scores > z_threshold) & series.notna()
            all_outliers |= outliers
            col_details.append(f"{col}: {int(outliers.sum())} outliers (|z|>{z_threshold})")

        score = 1 - (all_outliers.sum() / len(self.df))
        self.flags['accuracy'] = all_outliers
        self.results['accuracy'] = {
            'score'  : round(score, 4),
            'issues' : int(all_outliers.sum()),
            'details': " | ".join(col_details) if col_details else "No outliers detected"
        }
        return self

    # ── SCORECARD ─────────────────────────────────────────────────────────
    def scorecard(self) -> pd.DataFrame:
        """Returns a summary DataFrame of all assessed dimensions."""
        rows = []
        for dim, res in self.results.items():
            emoji = '🟢' if res['score'] >= 0.95 else ('🟡' if res['score'] >= 0.80 else '🔴')
            rows.append({
                'Table'    : self.table_name,
                'Dimension': dim.capitalize(),
                'Score'    : res['score'],
                'Issues'   : res['issues'],
                'Status'   : emoji,
                'Details'  : res['details']
            })
        return pd.DataFrame(rows)

    def overall_score(self) -> float:
        """Returns the average score across all evaluated dimensions."""
        if not self.results:
            return 0.0
        return round(np.mean([v['score'] for v in self.results.values()]), 4)

    def flag_rows(self) -> pd.DataFrame:
        """Returns combined flag DataFrame — feeds directly into cleaning pipeline."""
        return pd.DataFrame(self.flags, index=self.df.index)

    def plot_scorecard(self, save_path: str = None):
        """Visualizes dimension scores as a horizontal bar chart."""
        sc     = self.scorecard()
        colors = ['#2ecc71' if s >= 0.95 else ('#f39c12' if s >= 0.80 else '#e74c3c')
                  for s in sc['Score']]

        fig, ax = plt.subplots(figsize=(9, max(3, len(sc) * 0.7)))
        bars = ax.barh(sc['Dimension'], sc['Score'], color=colors,
                       edgecolor='white', height=0.5)
        ax.axvline(0.95, color='green',  linestyle='--', linewidth=1.5,
                   alpha=0.7, label='Target ≥ 0.95')
        ax.axvline(0.80, color='orange', linestyle='--', linewidth=1.5,
                   alpha=0.7, label='Warning ≥ 0.80')
        ax.set_xlim(0, 1.1)
        ax.set_xlabel('DQ Score (0–1)')
        ax.set_title(f'Data Quality Scorecard — {self.table_name}\n'
                     f'Overall: {self.overall_score():.2%}', fontweight='bold')
        ax.legend(fontsize=8, loc='lower right')

        # Add score labels on each bar
        for bar, score in zip(bars, sc['Score']):
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f'{score:.2%}', va='center', fontsize=9)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        overall = self.overall_score()
        status  = '🟢 GREEN' if overall >= 0.95 else ('🟡 YELLOW' if overall >= 0.80 else '🔴 RED')
        print(f"\n🏆 Overall Score — {self.table_name}: {overall:.2%}  {status}")

print("✅ DQAReport class defined — 6 dimensions ready")

✅ DQAReport class defined — 6 dimensions ready


### 3.2 DQA Pipeline Function

A single reusable entry point for running the full DQA on any table,
following the pattern introduced in the course materials (Oro, 2025).

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# run_dqa_pipeline — single entry point for repeatable DQA
# Adapted from course Colab 1 (Oro, 2025)
# ─────────────────────────────────────────────────────────────────────────────

def run_dqa_pipeline(df, table_name, pk,
                     completeness_cols,
                     uniqueness_cols,
                     validity_rules,
                     consistency_rules,
                     date_col=None,
                     max_age_days=1500,
                     allow_future=True,
                     numeric_cols=None,
                     z_threshold=3.0):
    """
    Single entry point for structured, repeatable DQA.
    Runs all 6 ISO 25012 dimensions and returns a DQAReport object.
    
    Parameters:
        df                : source DataFrame
        table_name        : name of the table being assessed
        pk                : primary key column name
        completeness_cols : list of columns that must be non-null
        uniqueness_cols   : list of columns forming the unique key
        validity_rules    : dict of {col: lambda Series -> bool Series}
        consistency_rules : list of lambdas df -> bool Series
        date_col          : column to check for timeliness (optional)
        max_age_days      : max allowed age in days (default 1500 for 2022-2024 data)
        allow_future      : whether to allow future dates (default True for historical data)
        numeric_cols      : list of numeric columns for accuracy check (optional)
        z_threshold       : z-score threshold for outlier detection (default 3.0)
    
    Returns:
        DQAReport object with all results and flags populated
    """

    # Initialize the report
    report = DQAReport(df, table_name=table_name, primary_key=pk)

    # Run all dimensions
    report.check_completeness(completeness_cols)
    report.check_uniqueness(uniqueness_cols)
    report.check_validity(validity_rules)
    report.check_consistency(consistency_rules)

    # Timeliness — only if date column provided
    if date_col:
        report.check_timeliness(date_col,
                                max_age_days=max_age_days,
                                allow_future=allow_future)

    # Accuracy — only if numeric columns provided
    if numeric_cols:
        report.check_accuracy(numeric_cols, z_threshold=z_threshold)

    return report

print("✅ run_dqa_pipeline() function defined")

✅ run_dqa_pipeline() function defined


## 4. Data Cleaning Pipeline
### 4.1 Missing Values
### 4.2 Outliers
### 4.3 Standardization

## 5. Build Dimension Tables
*Phase 1 reference: Section 1.4 — Star Schema, attribute tree editing Step 1 (pruning)*  
Column selection follows the pruning decisions documented in the attribute tree editing steps.

## 6. Build Fact Table
*Phase 1 reference: Section 1.4 — fact\_trips definition and glossary of measures*

### 6.1 Base joins
### 6.2 Pivot delivery\_events
*Phase 1 reference: Attribute tree editing Step 2 — pivoting and grafting of delivery\_events*

### 6.3 Aggregate fuel\_purchases
*Phase 1 reference: Attribute tree editing Step 4 — aggregation and grafting of fuel\_purchases*

### 6.4 Add dim\_date
*Phase 1 reference: Attribute tree editing Step 6 — promoting dispatch\_date to dim\_date*

### 6.5 Derived columns
*Phase 1 reference: Section 1.5 — Glossary of measures (derived measures section)*

---
# LOAD
---
## 7. Export to CSV
Export all final dimension and fact tables to `data/output/`.

---
## 8. Validation & Sanity Checks
Row count verification, referential integrity checks, and KPI spot checks.